### Interactive Photography Simulator

This widget demonstrates the exposure triangle:
- **F-Stop (Aperture):** A lower number means a wider aperture (more light). Higher numbers mean less light.
- **Exposure Time (Shutter Speed):** Longer exposure means more light enters the camera.
- **ISO:** Higher ISO makes the sensor more sensitive to light, resulting in a brighter image, but introduces digital noise (grain).

In [6]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from skimage import data

# 1. Load a real moon image using scikit-image
img_raw = data.moon()
base_moon = img_raw.astype(float) / 255.0

def simulate_photo(f_stop, exposure_time, iso):
    # Calculate exposure factor relative to a sunny day baseline:
    # Baseline: f/8, 1/125s, ISO 100
    base_exposure = (1/125 * 100) / (8**2)
    current_exposure = (exposure_time * iso) / (f_stop**2)

    exposure_multiplier = current_exposure / base_exposure

    # Apply exposure to the image (0.4 is a mid-gray baseline)
    img = base_moon * exposure_multiplier * 0.4

    # Simulate ISO noise (higher ISO = more noise)
    noise_amplitude = (iso / 100) * 0.02
    # Make sure noise shape matches the actual loaded image shape
    noise = np.random.randn(*base_moon.shape) * noise_amplitude
    img = img + noise

    # Clip values to valid image display range [0, 1]
    img = np.clip(img, 0, 1)

    # Plotting
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # Image subplot
    ax1.imshow(img, cmap='gray', vmin=0, vmax=1)
    ax1.axis('off')
    ax1.set_title(f"Simulated Moon Image\nf/{f_stop} | {exposure_time}s | ISO {iso}", fontsize=14)

    # Histogram subplot
    ax2.hist(img.ravel(), bins=50, range=(0, 1), color='gray')
    ax2.set_title("Brightness Histogram", fontsize=14)
    ax2.set_xlabel("Pixel Brightness (0=Black, 1=White)")
    ax2.set_ylabel("Frequency")
    ax2.set_xlim(0, 1)

    plt.tight_layout()
    plt.show()

# 2. Create interactive widgets
style = {'description_width': 'initial'}

f_stop_slider = widgets.SelectionSlider(
    options=[1.4, 2, 2.8, 4, 5.6, 8, 11, 16, 22],
    value=8,
    description='F-Stop (Aperture):',
    style=style
)

exposure_slider = widgets.SelectionSlider(
    options=[
        ('1/4000', 1/4000), ('1/2000', 1/2000), ('1/1000', 1/1000),
        ('1/500', 1/500), ('1/250', 1/250), ('1/125', 1/125),
        ('1/60', 1/60), ('1/30', 1/30), ('1/15', 1/15),
        ('1/8', 1/8), ('1/4', 1/4), ('1/2', 1/2), ('1s', 1.0)
    ],
    value=1/125,
    description='Exposure Time (s):',
    style=style
)

iso_slider = widgets.SelectionSlider(
    options=[100, 200, 400, 800, 1600, 3200, 6400],
    value=100,
    description='ISO:',
    style=style
)

reset_button = widgets.Button(
    description='Reset to Defaults',
    button_style='info',
    tooltip='Reset sliders to initial values'
)

def reset_values(b):
    f_stop_slider.value = 8
    exposure_slider.value = 1/125
    iso_slider.value = 100

reset_button.on_click(reset_values)

# 3. Link widgets to the function and display side-by-side
output = widgets.interactive_output(simulate_photo, {
    'f_stop': f_stop_slider,
    'exposure_time': exposure_slider,
    'iso': iso_slider
})

controls = widgets.VBox([f_stop_slider, exposure_slider, iso_slider, reset_button], layout=widgets.Layout(padding='20px'))
ui = widgets.HBox([controls, output], layout=widgets.Layout(align_items='center'))

display(ui)